In [1]:
%%capture
!pip install timm

In [2]:
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
from pathlib import Path
from PIL import Image
from urllib.request import urlopen
from torchvision.ops import roi_align
from torch.utils.data import DataLoader, Dataset
import random

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [ ]:
class FPNWrapper(nn.Module):
    def __init__(self, backbone, out_channels=256):
        super().__init__()
        # Assume backbone returns feature maps from 3 stages
        self.backbone = backbone
        self.lateral3 = nn.Conv2d(48, out_channels, 1)
        self.lateral4 = nn.Conv2d(80, out_channels, 1)
        self.lateral5 = nn.Conv2d(960, out_channels, 1)
        self.smoothing3 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.smoothing4 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.smoothing5 = nn.Conv2d(out_channels, out_channels, 3, padding=1)

    def forward(self, x):
        # Example: features from conv3, conv4, conv5
        c3, c4, c5 = self.backbone(x)
        p5 = self.lateral5(c5)
        p5 = self.smoothing5(p5)
        p4 = self.lateral4(c4) + F.interpolate(p5, size=c4.shape[-2:], mode='nearest')
        p4 = self.smoothing4(p4)
        p3 = self.lateral3(c3) + F.interpolate(p4, size=c3.shape[-2:], mode='nearest')
        p3 = self.smoothing3(p3)
        return [p3, p4, p5]

class AttentionPooling(nn.Module):
    def __init__(self, in_channels=256):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 8, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // 8, 1, 1),
            nn.Sigmoid()
        )

    def forward(self, feat):
        attn_map = self.attn(feat)
        weighted = (feat * attn_map).sum(dim=(2, 3)) / (attn_map.sum(dim=(2, 3)) + 1e-6)
        return weighted, attn_map

class PrototypeExtractor(nn.Module):
    def __init__(self, fpn, attn_pool, output_size=(7, 7)):
        super().__init__()
        self.fpn = fpn
        self.attn_pool = attn_pool
        self.output_size = output_size

    def forward(self, support_patches: torch.Tensor):


        # Safety check for empty input
        if support_patches.size(0) == 0:
            return None


        fpn_feats = self.fpn(support_patches)

        roi_feat = fpn_feats[-1]


        if roi_feat.shape[-2:] != self.output_size:
             roi_feat = F.adaptive_avg_pool2d(roi_feat, self.output_size)


        proto, _ = self.attn_pool(roi_feat)


        return proto

class Backbone(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model


    def forward(self, x):
        # Return feature maps from 3 stages
        c3, c4, c5 = self.model(x)
        return c3, c4, c5

    def destroy_hooks(self):
        # Call this method to remove hooks when done
        self.handle1.remove()
        self.handle2.remove()
        self.handle3.remove()

In [4]:
def calculate_iou(boxA, boxB):
    # ... (implementation from above) ...
    inter_x1 = max(boxA[0], boxB[0])
    inter_y1 = max(boxA[1], boxB[1])
    inter_x2 = min(boxA[2], boxB[2])
    inter_y2 = min(boxA[3], boxB[3])
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    if inter_area == 0: return 0.0
    area_A = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    area_B = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    union_area = area_A + area_B - inter_area
    return inter_area / union_area

import torch
from torch.utils.data import Dataset
from PIL import Image
import random

class ContrastiveDetectionDataset(Dataset):
    def __init__(self, support_data, query_data, transform,
                 bg_iou_threshold=0.2, max_bg_attempts=10):

        self.support_data = support_data
        self.query_data = query_data
        self.transform = transform
        self.bg_iou_threshold = bg_iou_threshold
        self.max_bg_attempts = max_bg_attempts

        # --- Pre-process query data ---
        self.flat_query_annotations = []
        self.query_images_by_path = {}

        for query_img_info in self.query_data:
            img_path = query_img_info['img_path']
            annotations = query_img_info['annotations']

            if img_path not in self.query_images_by_path:
                self.query_images_by_path[img_path] = []

            for ann in annotations:
                self.flat_query_annotations.append({
                    'img_path': img_path,
                    'box': ann['box'],
                    'label': ann['label']
                })
                self.query_images_by_path[img_path].append(ann)

        self.all_labels = list(support_data.keys())

    def __len__(self):
        return len(self.flat_query_annotations)

    # ### FIX: Added IoU Helper function
    @staticmethod
    def calculate_iou(box1, box2):
        # Box format: [x1, y1, x2, y2]
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])

        intersection = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
        area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])

        union = area1 + area2 - intersection
        return intersection / (union + 1e-6)

    def __getitem__(self, index):

        # 1. Get Anchor
        anchor_info = self.flat_query_annotations[index]
        anchor_label = anchor_info['label']
        anchor_img_path = anchor_info['img_path']
        anchor_box = anchor_info['box']

        # 2. Get Reference (Support)
        # WARNING: Ensure these are CROPPED images of the object.
        # If these are full images, your model will fail to converge.
        ref_path = random.choice(self.support_data[anchor_label])
        ref_image = Image.open(ref_path).convert('RGB')

        # 3. Decide Pair Type
        if random.random() < 0.5:
            # --- POSITIVE PAIR ---
            label = 1.0
            search_image = Image.open(anchor_img_path).convert('RGB')
            search_patch = search_image.crop(anchor_box)
        else:
            # --- NEGATIVE PAIR ---
            label = 0.0
            search_image = None

            # 50% chance of Hard Background vs Different Class
            if random.random() < 0.5:
                # --- Type B: Cross-Class Negative ---
                while True:
                    neg_info = random.choice(self.flat_query_annotations)
                    if neg_info['label'] != anchor_label:
                        break

                search_image = Image.open(neg_info['img_path']).convert('RGB')
                search_patch = search_image.crop(neg_info['box'])

            else:
                # --- Type A: Hard Background Negative ---
                search_image = Image.open(anchor_img_path).convert('RGB')
                img_w, img_h = search_image.size

                # ### FIX: Ensure these are integers for random.randint
                all_gt_boxes = [ann['box'] for ann in self.query_images_by_path[anchor_img_path]]

                anchor_w = int(anchor_box[2] - anchor_box[0])
                anchor_h = int(anchor_box[3] - anchor_box[1])

                found_negative = False
                for _ in range(self.max_bg_attempts):
                    # ### FIX: Cast max range to int to prevent TypeError
                    max_x = int(max(0, img_w - anchor_w - 1))
                    max_y = int(max(0, img_h - anchor_h - 1))

                    # If image is smaller than crop (edge case), break
                    if max_x == 0 or max_y == 0:
                        break

                    neg_x1 = random.randint(0, max_x)
                    neg_y1 = random.randint(0, max_y)

                    neg_x2 = neg_x1 + anchor_w
                    neg_y2 = neg_y1 + anchor_h
                    neg_box = [neg_x1, neg_y1, neg_x2, neg_y2]

                    # Check IoU
                    max_iou = 0.0
                    for gt_box in all_gt_boxes:
                        iou = self.calculate_iou(neg_box, gt_box)
                        max_iou = max(max_iou, iou)

                    if max_iou <= self.bg_iou_threshold:
                        search_patch = search_image.crop(neg_box)
                        found_negative = True
                        break

                # Fallback if background mining failed
                if not found_negative:
                    while True:
                        neg_info = random.choice(self.flat_query_annotations)
                        if neg_info['label'] != anchor_label:
                            break
                    search_image = Image.open(neg_info['img_path']).convert('RGB')
                    search_patch = search_image.crop(neg_info['box'])

        # 4. Transforms
        if self.transform:
            ref_image = self.transform(ref_image)
            search_patch = self.transform(search_patch)

        return ref_image, search_patch, torch.tensor(label, dtype=torch.float32)


### Process data

In [5]:
def xywh_norm_to_xyxy_abs(bbox_norm, img_width = 1024, img_height=576):
    # Unpack normalized center coordinates and dimensions
    norm_cx, norm_cy, norm_w, norm_h = bbox_norm

    # Convert to absolute pixel values
    abs_cx = round(norm_cx * img_width, 3)
    abs_cy = round(norm_cy * img_height, 3)
    abs_w = round(norm_w * img_width, 3)
    abs_h = round(norm_h * img_height, 3)

    # Convert from center (x,y) to top-left (x,y)
    abs_x = abs_cx - (abs_w / 2)
    abs_y = abs_cy - (abs_h / 2)

    return [abs_x, abs_y, abs_x + abs_w, abs_y + abs_h]

def str2list(s: str, round_values: bool = False):
    tokens = s.strip().split()
    return [float(t) for t in tokens]

In [6]:
from collections import defaultdict
gd_img_dict = {}
idx = 0
support_folder = "/kaggle/input/zaloai2025-aeroeyes/observing/train/samples/"
for fol in os.listdir(support_folder):
    gd_img_dict[idx] = []
    for img_name in os.listdir(support_folder + fol+"/object_images/"):
        img_path =support_folder + fol+"/object_images/" + img_name
        gd_img_dict[idx].append(img_path)
    idx += 1

img_folder = "/kaggle/input/zaloai-aero/data_one_class/data_one_class/train/images/"
ground_truth_folder = "/kaggle/input/zaloai-aero/labels_n_class/labels/"
ground_truth_labels = os.listdir(ground_truth_folder)
query_data = []
for gd_label in ground_truth_labels:
    # image_id = gd_label.split('.')[0]
    with open(ground_truth_folder + "/" + gd_label, "r") as f:
            gd_bboxs = [str2list(a) for a in f.readlines()]
    img_path = img_folder + "/" + gd_label.split(".")[0] + ".jpg"
    gd_bboxs_dict = {
         "img_path": img_path,
         "annotations": [],
    }
    for gd_bbox in gd_bboxs:
         gd_bboxs_dict["annotations"].append({
             "box": xywh_norm_to_xyxy_abs(gd_bbox[1:]),
             "label": int(gd_bbox[0])
         })
    query_data.append(gd_bboxs_dict)
    # gd_bboxs_dict["img_path "] = {"bboxs": [], "category_id": 0}
    # gd_bboxs_dict['bboxs'] = [xywh_norm_to_xyxy_abs(bbox[1:]) for bbox in gd_bboxs]
    # gd_bboxs_dict['category_id'] = [bbox[0] for bbox in gd_bboxs]

In [7]:
len(query_data)

5413

In [ ]:
from torch.utils.data import random_split
model = timm.create_model(
    "hf_hub:timm/mobilenetv4_conv_medium.e500_r224_in1k",              
      pretrained=True,
      features_only=True,
      out_indices=(1, 2, 4)  # Number of output classes
  ).to("cpu")
model.eval()

data_config = timm.data.resolve_model_data_config(model)
transforms = timm.data.create_transform(**data_config, is_training=False)

len_train = int(0.9 * len(query_data))

t_set, v_set = random_split(query_data, [len_train, len(query_data) - len_train])
train_set = ContrastiveDetectionDataset(gd_img_dict, t_set, transform=transforms)
val_set = ContrastiveDetectionDataset(gd_img_dict, v_set, transform=transforms)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=64, shuffle=True)
backbone = Backbone(model)
fpn = FPNWrapper(backbone)
att = AttentionPooling()
prototypeExtractor = PrototypeExtractor(fpn, att)


config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/39.2M [00:00<?, ?B/s]

Unexpected keys (classifier.bias, classifier.weight, conv_head.weight, norm_head.bias, norm_head.num_batches_tracked, norm_head.running_mean, norm_head.running_var, norm_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


In [9]:
import wandb
import torch
import torch.nn as nn
from tqdm import tqdm
import os
from torch.optim.lr_scheduler import CosineAnnealingLR

# --- SECURITY WARNING ---
# Do not hardcode API keys in scripts. Use 'wandb login' in terminal
# or set the WANDB_API_KEY environment variable.
# wandb.login(key="YOUR_KEY_HERE")
wandb.login(key="01b213357e510532f2fe0b4165a6a8517b077956")
# 1. Configuration
num_epochs = 30
lr = 3e-4
batch_size = 1  # If this is >1, the code below still works perfectly
best_val_loss = float('inf') # To track the best model
min_delta = 1e-4
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# Initialize WandB
run = wandb.init(
    project="prototype-siamese-training",
    name="experiment-with-validation",
    config={
        "learning_rate": lr,
        "epochs": num_epochs,
        "batch_size": batch_size,
        "architecture": "MobileNetV4-FPN",
        "margin": 0.5,
        "min_delta": min_delta
    }
)

# Setup Model & Loss
# Assuming 'prototypeExtractor' is defined elsewhere
# loss_fn = nn.CosineEmbeddingLoss(margin=wandb.config.margin)
# optimizer = torch.optim.Adam(prototypeExtractor.parameters(), lr=wandb.config.learning_rate)
loss_fn = nn.CosineEmbeddingLoss()
optimizer = torch.optim.Adam(prototypeExtractor.parameters(), lr=lr)


# 1. Initialize Model & Optimizer as usual
# (Make sure you define them exactly as you did in the training script)
optimizer = torch.optim.Adam(prototypeExtractor.parameters(), lr=lr) 

# 2. Load the Checkpoint
start_epoch = 0  # Default start if no checkpoint found
checkpoint_path = "/kaggle/input/siamese-model/pytorch/default/3/last.pth"

if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path)
    
    # A. Load Model Weights
    prototypeExtractor.load_state_dict(checkpoint['model_state_dict'])
    prototypeExtractor.to(device)
    # B. Load Optimizer State (Momentum, etc.)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    start_epoch = checkpoint['epoch'] + 1
    for param_group in optimizer.param_groups:
        if "initial_lr" not in param_group:
            param_group["initial_lr"] = 5e-4
    
    print(f"Resuming training from Epoch {start_epoch}")
else:
    print("No checkpoint found. Starting from scratch.")

scheduler = CosineAnnealingLR(
    optimizer, 
    T_max=num_epochs, 
    last_epoch=start_epoch - 1  # -1 tells it we are done with previous epochs
)

patience = 12
current_patient = 0

# Create directory for checkpoints
os.makedirs("checkpoints", exist_ok=True)

# 2. Training Loop
global_step = 0

for epoch in range(start_epoch, num_epochs):

    # ==========================
    #       TRAINING PHASE
    # ==========================
    prototypeExtractor.train()

    train_loss = 0.0
    train_pos_sim = 0.0
    train_neg_sim = 0.0
    train_pos_count = 0
    train_neg_count = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")

    for batch in pbar:
        input1 = batch[0].to(device)
        input2 = batch[1].to(device)
        label = batch[2].float().to(device)
        # Target for Loss (-1/1)
        target = torch.where(label == 0, torch.tensor(-1.0).to(device), label)

        # Forward pass
        proto_a = prototypeExtractor(input1)
        proto_b = prototypeExtractor(input2)

        loss = loss_fn(proto_a, proto_b, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        # --- FIX: Vectorized Metrics (No more loops) ---
        with torch.no_grad():
            # 1. Calculate Cosine Similarity
            # .view(-1) ensures we have a flat vector, even if batch_size=1
            sims = torch.cosine_similarity(proto_a, proto_b, dim=1).view(-1)
            label_flat = label.view(-1)

            # 2. Create Boolean Masks
            pos_mask = (label_flat == 1.0)
            neg_mask = (label_flat == 0.0) # or ~pos_mask if labels are strictly 0/1

            # 3. Aggregate (Sum) based on masks
            # We use checks to avoid summing empty tensors if a batch has no pos or no neg samples
            if pos_mask.any():
                train_pos_sim += sims[pos_mask].sum().item()
                train_pos_count += pos_mask.sum().item()

            if neg_mask.any():
                train_neg_sim += sims[neg_mask].sum().item()
                train_neg_count += neg_mask.sum().item()

        # Log Step Metrics
        wandb.log({
            "train/step_loss": loss.item(),
            "global_step": global_step
        })

        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        global_step += 1
    scheduler.step()
    # ==========================
    #      VALIDATION PHASE
    # ==========================
    prototypeExtractor.eval()

    val_loss = 0.0
    val_pos_sim = 0.0
    val_neg_sim = 0.0
    val_pos_count = 0
    val_neg_count = 0

    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]")

        for batch in val_pbar:
            input1 = batch[0].to(device)
            input2 = batch[1].to(device)
            label = batch[2].float().to(device)
            target = torch.where(label == 0, torch.tensor(-1.0).to(device), label)

            proto_a = prototypeExtractor(input1)
            proto_b = prototypeExtractor(input2)

            loss = loss_fn(proto_a, proto_b, target)
            val_loss += loss.item()

            # --- FIX: Same Vectorized Logic for Validation ---
            sims = torch.cosine_similarity(proto_a, proto_b, dim=1).view(-1)
            label_flat = label.view(-1)

            pos_mask = (label_flat == 1.0)
            neg_mask = (label_flat == 0.0)

            if pos_mask.any():
                val_pos_sim += sims[pos_mask].sum().item()
                val_pos_count += pos_mask.sum().item()

            if neg_mask.any():
                val_neg_sim += sims[neg_mask].sum().item()
                val_neg_count += neg_mask.sum().item()
    # ==========================
    #    EPOCH AGGREGATION
    # ==========================

    # Calculate Train Averages
    avg_train_loss = train_loss / len(train_loader)
    avg_train_pos = train_pos_sim / (train_pos_count + 1e-6)
    avg_train_neg = train_neg_sim / (train_neg_count + 1e-6)

    # Calculate Val Averages
    avg_val_loss = val_loss / len(val_loader)
    avg_val_pos = val_pos_sim / (val_pos_count + 1e-6)
    avg_val_neg = val_neg_sim / (val_neg_count + 1e-6)

    print(f"\nEpoch {epoch+1} Summary:")
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"Val Pos Sim: {avg_val_pos:.4f} | Val Neg Sim: {avg_val_neg:.4f}")

    if avg_val_loss > best_val_loss:
        current_patient += 1
        if current_patient >= patience:
            print(f"Early stopping triggered after {patience} epochs without improvement.")
            break
    else:
        current_patient = 0

    # Log Epoch Metrics to WandB
    wandb.log({
        "epoch": epoch + 1,
        "train/epoch_loss": avg_train_loss,
        "train/avg_pos_sim": avg_train_pos,
        "train/avg_neg_sim": avg_train_neg,
        "val/epoch_loss": avg_val_loss,
        "val/avg_pos_sim": avg_val_pos,
        "val/avg_neg_sim": avg_val_neg,
    })

    # ==========================
    #      CHECKPOINTING
    # ==========================

    # 1. Save latest model
    ckpt = {
        'epoch': epoch,
        'model_state_dict': prototypeExtractor.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': avg_train_loss,
        'val_loss': avg_val_loss
    }
    torch.save(ckpt, "checkpoints/last.pth")

    # 2. Save best model (if validation loss improved)
    if avg_val_loss < best_val_loss - min_delta:
        best_val_loss = avg_val_loss
        current_patient = 0
        torch.save(ckpt, "checkpoints/best.pth")
        print(f"--> New Best Model Saved! (Val Loss: {best_val_loss:.4f})")
    else:
        current_patient += 1
        if current_patient >= patience:
            print(f"Early stopping triggered after {patience} epochs without improvement.")
            break

# wandb.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: onghoangcodebug01 (onghoangcodebug01-tr-ng-i-h-c-khoa-h-c-t-nhi-n-hqg-hcm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.21.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20251119_141129-1ii1zpn4
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run experiment-with-validation
wandb: ⭐️ View project at https://wandb.ai/onghoangcodebug01-tr-ng-i-h-c-khoa-h-c-t-nhi-n-hqg-hcm/prototype-siamese-training
wandb: 🚀 View run at https://wandb.ai/onghoangcodebug01-tr-ng-i-h-c-khoa-h-c-t-nhi-n-hqg-hcm/prototype-siamese-training/runs/1ii1zpn4


Loading checkpoint from /kaggle/input/siamese-model/pytorch/default/3/last.pth...
Resuming training from Epoch 6


Epoch 7/30 [Val]: 100%|██████████| 9/9 [01:50<00:00, 12.26s/it]



Epoch 7 Summary:
Train Loss: 0.0844 | Val Loss: 0.3062
Val Pos Sim: 0.5491 | Val Neg Sim: 0.0143
--> New Best Model Saved! (Val Loss: 0.3062)


Epoch 8/30 [Val]: 100%|██████████| 9/9 [01:36<00:00, 10.67s/it]


...


Epoch 23 Summary:
Train Loss: 0.0523 | Val Loss: 0.2743
Val Pos Sim: 0.6693 | Val Neg Sim: 0.0762


Epoch 24/30 [Val]: 100%|██████████| 9/9 [01:55<00:00, 12.86s/it]



Epoch 24 Summary:
Train Loss: 0.0515 | Val Loss: 0.2804
Val Pos Sim: 0.6570 | Val Neg Sim: 0.0961


Epoch 25/30 [Val]: 100%|██████████| 9/9 [01:47<00:00, 11.94s/it]



Epoch 25 Summary:
Train Loss: 0.0533 | Val Loss: 0.2491
Val Pos Sim: 0.6432 | Val Neg Sim: 0.0123
Early stopping triggered after 12 epochs without improvement.
